# TradeStation SDK - Market Data

This notebook demonstrates how to:
- Get historical bars (OHLCV data)
- Search for symbols
- Get real-time quote snapshots
- Get symbol details
- Visualize market data with charts

## Prerequisites

1. Complete `01_authentication.ipynb` first
2. Have valid TradeStation authentication
3. Install visualization libraries: `pip install matplotlib pandas`

In [ ]:
# Import libraries
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from tradestation_sdk import TradeStationSDK

# Load environment variables
load_dotenv()

# Initialize and authenticate
sdk = TradeStationSDK()
sdk.ensure_authenticated(mode="PAPER")
print("✅ SDK initialized and authenticated")

## 1. Get Historical Bars

Historical bars provide OHLCV (Open, High, Low, Close, Volume) data for technical analysis.

In [ ]:
# Get 100 1-minute bars for AAPL
bars = sdk.get_bars(symbol="AAPL", interval="1", unit="Minute", bars_back=100, mode="PAPER")

print(f"Retrieved {len(bars)} bars")

# Show last 5 bars
print("\nLast 5 bars:")
for bar in bars[-5:]:
    print(
        f"{bar['TimeStamp']}: O={bar['Open']:.2f}, H={bar['High']:.2f}, L={bar['Low']:.2f}, C={bar['Close']:.2f}, V={bar['Volume']}"
    )

## 2. Visualize Bars with Chart

In [ ]:
# Convert to DataFrame for easy plotting
df = pd.DataFrame(bars)
df["TimeStamp"] = pd.to_datetime(df["TimeStamp"])
df = df.set_index("TimeStamp")

# Plot close prices
plt.figure(figsize=(12, 6))
plt.plot(df.index, df["Close"], label="Close", linewidth=2)
plt.fill_between(df.index, df["Low"], df["High"], alpha=0.3, label="High-Low Range")
plt.title("AAPL - 1-Minute Bars (Last 100)", fontsize=14, fontweight="bold")
plt.xlabel("Time")
plt.ylabel("Price ($)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 Chart displayed above")

## 3. Search for Symbols

Search for stocks, futures, options, or other instruments.

In [ ]:
# Search for MNQ futures contracts
symbols = sdk.search_symbols(pattern="MNQ", category="Future", asset_type="Index", mode="PAPER")

print(f"Found {len(symbols)} MNQ contracts:\n")
for symbol in symbols[:5]:  # Show first 5
    print(f"  {symbol['Symbol']}: {symbol.get('Description', 'N/A')}")

## 4. Get Real-Time Quote Snapshots

Get current bid, ask, last price for multiple symbols.

In [ ]:
# Get quotes for multiple symbols (comma-separated)
quotes = sdk.get_quote_snapshots("AAPL,MSFT,GOOGL", mode="PAPER")

print("📈 Current Quotes:\n")
for quote in quotes["Quotes"]:
    symbol = quote["Symbol"]
    last = quote.get("Last", "N/A")
    bid = quote.get("Bid", "N/A")
    ask = quote.get("Ask", "N/A")
    spread = float(ask) - float(bid) if ask != "N/A" and bid != "N/A" else "N/A"

    print(f"{symbol:6} - Last: ${last:>8}, Bid: ${bid:>8}, Ask: ${ask:>8}, Spread: ${spread}")

## 5. Get Symbol Details

Get detailed information about a symbol (exchange, category, etc.).

In [ ]:
# Get symbol details
details = sdk.get_symbol_details("AAPL", mode="PAPER")

if details.get("Symbols"):
    symbol_info = details["Symbols"][0]
    print(f"Symbol: {symbol_info['Symbol']}")
    print(f"Description: {symbol_info.get('Description', 'N/A')}")
    print(f"Category: {symbol_info.get('Category', 'N/A')}")
    print(f"Exchange: {symbol_info.get('Exchange', 'N/A')}")
    print(f"Asset Type: {symbol_info.get('AssetType', 'N/A')}")

## 6. Multi-Timeframe Analysis

Compare price action across different timeframes.

In [ ]:
# Get bars at different intervals
bars_1min = sdk.get_bars("AAPL", "1", "Minute", 100, mode="PAPER")
bars_5min = sdk.get_bars("AAPL", "5", "Minute", 100, mode="PAPER")
bars_15min = sdk.get_bars("AAPL", "15", "Minute", 100, mode="PAPER")

# Convert to DataFrames
df_1min = pd.DataFrame(bars_1min)
df_5min = pd.DataFrame(bars_5min)
df_15min = pd.DataFrame(bars_15min)

# Plot all timeframes
fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# 1-minute
axes[0].plot(pd.to_datetime(df_1min["TimeStamp"]), df_1min["Close"], label="1-Min", color="blue")
axes[0].set_title("AAPL - 1-Minute Bars")
axes[0].set_ylabel("Price ($)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# 5-minute
axes[1].plot(pd.to_datetime(df_5min["TimeStamp"]), df_5min["Close"], label="5-Min", color="green")
axes[1].set_title("AAPL - 5-Minute Bars")
axes[1].set_ylabel("Price ($)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# 15-minute
axes[2].plot(pd.to_datetime(df_15min["TimeStamp"]), df_15min["Close"], label="15-Min", color="red")
axes[2].set_title("AAPL - 15-Minute Bars")
axes[2].set_ylabel("Price ($)")
axes[2].set_xlabel("Time")
axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.show()

print("\n📊 Multi-timeframe chart displayed above")

## Summary

You've successfully:
- ✅ Retrieved historical bars
- ✅ Searched for symbols
- ✅ Got real-time quotes
- ✅ Retrieved symbol details
- ✅ Visualized market data with charts
- ✅ Performed multi-timeframe analysis

## Next Steps

- 💼 [04_placing_orders.ipynb](04_placing_orders.ipynb) - Learn to place orders
- 📊 [05_streaming_data.ipynb](05_streaming_data.ipynb) - Real-time data streaming
- 🎯 [06_bracket_orders.ipynb](06_bracket_orders.ipynb) - Advanced bracket orders